# Week 09, Quantization Lab: shrink the triage model, keep the skill

# Requirements: pip install torch transformers accelerate bitsandbytes numpy pandas

# ⚠️ REQUIRES: NVIDIA GPU or Apple Silicon for the live cells; CPU fallback included

This notebook loads a small instruct model (`Qwen/Qwen2.5-1.5B-Instruct`) in three
precisions and measures two things per precision: **perplexity** on a sample of
shipment notes, and **triage accuracy** on 30 labeled support tickets. The result is a
quality-vs-VRAM table, the input to your Friday report. On a CUDA GPU you get live
8-bit/4-bit numbers; on Apple Silicon you get live FP16 (bitsandbytes does not support
MPS); on CPU only, the last cells produce the documented quality ladder as a clearly
labeled *estimate* so the notebook still ends in a number.


## 0. Setup: repo root on the path + seeded RNG

Every notebook in this program starts the same way: put the repo root on `sys.path`
so `from zoro import data` works, and seed every random draw.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parents[1]))  # repo root

import os, time, math, json
import numpy as np
import pandas as pd

from zoro import data

SEED = 42
np.random.seed(SEED)

try:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
    _HF_OK = True
except Exception as _e:  # pragma: no cover
    torch = None
    _HF_OK = False
    print("transformers/torch not available; CPU fallback will run instead:", _e)

if torch is not None:
    torch.manual_seed(SEED)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)

print("imports ok, HF available:", _HF_OK)


## 1. Detect the accelerator

The lab degrades in three steps, and it says so loudly instead of crashing:
CUDA runs everything live; MPS runs FP16 only (bitsandbytes is CUDA-only); CPU skips
model loading entirely and uses the documented quality ladder.


In [ ]:
def detect_device():
    if torch is None:
        return "cpu"
    if torch.cuda.is_available():
        return "cuda"
    mps = getattr(torch.backends, "mps", None)
    if mps is not None and mps.is_available():
        return "mps"
    return "cpu"

DEVICE = detect_device()
LIVE_QUANT = (DEVICE == "cuda")   # bitsandbytes 8-bit / 4-bit needs CUDA
LIVE_FP16  = (DEVICE in ("cuda", "mps"))
print("Detected device:", DEVICE)
print("live FP16 cells:", LIVE_FP16, "| live 8/4-bit cells:", LIVE_QUANT)


## 2. The data: shipment notes + 30 labeled tickets

Perplexity is measured on 8 natural-language **shipment notes** built from
`data.shipments()` joined with `data.lanes()`. Triage accuracy is measured on 30
**labeled** tickets from `data.support_tickets()`, the ground-truth category is the
`category` column we are trying to recover.


In [ ]:
ship = data.shipments(n=2000, seed=SEED)
lan = data.lanes(n=20, seed=11)
notes_df = ship.merge(lan[["lane_id", "origin", "destination"]], on="lane_id", how="left")
notes_df = notes_df.dropna(subset=["origin", "destination"]).head(8)

def make_note(row):
    return (f"Shipment {row['shipment_id']} of {row['commodity']} ({row['weight_kg']} kg) "
            f"moved from {row['origin']} to {row['destination']} on lane {row['lane_id']}. "
            f"Status: {row['status']}. Weather: {row['weather_severity']}.")

shipment_notes = [make_note(r) for _, r in notes_df.iterrows()]
print(f"Loaded {len(shipment_notes)} shipment notes for perplexity:")
print(" -", shipment_notes[0])

tickets = data.support_tickets(n=30, seed=99)
print(f"\nLoaded {len(tickets)} labeled triage tickets.")
print(tickets["category"].value_counts().to_string())


## 3. VRAM math first (always runs)

Before touching a model, compute what each precision costs. The rule of thumb from
`reference/knowledge-base/06-model-engineering.md`: **FP16 ≈ 2 bytes/param, INT8 ≈ 1, INT4 ≈ 0.5**.
Qwen2.5-1.5B has ~1.54B parameters.


In [ ]:
def vram_bytes(num_params, bits):
    return num_params * (bits / 8)

PARAMS = 1.54e9  # Qwen2.5-1.5B-Instruct
print(f"{'precision':>9}  {'weights-only':>14}")
for bits, name in [(16, "FP16"), (8, "INT8"), (4, "INT4/NF4")]:
    gb = vram_bytes(PARAMS, bits) / 1e9
    print(f"{name:>9}  {gb:>12.2f} GB")


## 4. Load the FP16 baseline

`device_map="auto"` lets accelerate place layers for us. FP16 needs ~3 GB for weights,
comfortable on any recent GPU or an M-series Mac.


In [ ]:
MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"

model = None
tokenizer = None
if not _HF_OK or not LIVE_FP16:
    print("Skipping FP16 load, needs torch+transformers and CUDA or MPS.")
else:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, torch_dtype=torch.float16, device_map="auto", trust_remote_code=True
    )
    model.eval()
    print("Loaded FP16 model on", DEVICE)


## 5. Perplexity

Perplexity is `exp(cross-entropy loss)` over the note's own tokens, "how surprised is
the model by text it should find fluent." Lower is better. We average the per-token loss
across the notes so short and long notes contribute equally.


In [ ]:
def perplexity(model, tokenizer, texts, max_length=64):
    if model is None:
        return None
    total_loss, total_tokens = 0.0, 0
    model.eval()
    with torch.no_grad():
        for t in texts:
            enc = tokenizer(t, return_tensors="pt", truncation=True, max_length=max_length)
            enc = {k: v.to(model.device) for k, v in enc.items()}
            out = model(**enc, labels=enc["input_ids"])
            total_loss += out.loss.item() * enc["input_ids"].numel()
            total_tokens += enc["input_ids"].numel()
    return math.exp(total_loss / max(total_tokens, 1))

ppl_fp16 = perplexity(model, tokenizer, shipment_notes) if LIVE_FP16 else None
print("FP16 perplexity:", None if ppl_fp16 is None else round(ppl_fp16, 3))


## 6. 8-bit quantization (bitsandbytes `LLM.int8()`)

`load_in_8bit=True` is the one-line switch. Expect ~negligible perplexity change, the
interesting question is whether the *triage eval* agrees.


In [ ]:
model_8bit = None
if not LIVE_QUANT:
    print("Skipping 8-bit load, bitsandbytes needs a CUDA GPU.")
else:
    model_8bit = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, load_in_8bit=True, device_map="auto", trust_remote_code=True
    )
    model_8bit.eval()
    print("Loaded 8-bit model (LLM.int8())")


In [ ]:
ppl_8bit = perplexity(model_8bit, tokenizer, shipment_notes) if LIVE_QUANT else None
print("8-bit perplexity:", None if ppl_8bit is None else round(ppl_8bit, 3))


## 7. 4-bit NF4 quantization

NF4 ("normal-float-4") is a data-aware format tuned to the weight distribution; it is the
substrate of QLoRA (Week 10). Double quantization shaves the scaling constants too.


In [ ]:
model_4bit = None
if not LIVE_QUANT:
    print("Skipping 4-bit load, bitsandbytes needs a CUDA GPU.")
else:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_use_double_quant=True,
    )
    model_4bit = AutoModelForCausalLM.from_pretrained(
        MODEL_ID, quantization_config=bnb_config, device_map="auto", trust_remote_code=True
    )
    model_4bit.eval()
    print("Loaded 4-bit NF4 model")


In [ ]:
ppl_4bit = perplexity(model_4bit, tokenizer, shipment_notes) if LIVE_QUANT else None
print("4-bit perplexity:", None if ppl_4bit is None else round(ppl_4bit, 3))


## 8. The task eval: triage accuracy on 30 labeled tickets

Perplexity is the screen; this is the verdict. For each of the 30 tickets we ask the
model to output one category word, then compare against the ground-truth label. Greedy
decoding (`do_sample=False`) keeps the run deterministic.


In [ ]:
CATEGORIES = ["tracking", "damage", "refund", "documents", "customs", "billing"]

def triage_prompt(text):
    return (f"Classify this support ticket into exactly one category from "
            f"{CATEGORIES}. Reply with only the category word.\nTicket: {text}")

def predict_category(model, tokenizer, text):
    if model is None or tokenizer is None:
        return None
    prompt = triage_prompt(text)
    enc = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=256)
    enc = {k: v.to(model.device) for k, v in enc.items()}
    with torch.no_grad():
        gen = model.generate(**enc, max_new_tokens=8, do_sample=False,
                             pad_token_id=tokenizer.eos_token_id)
    out = tokenizer.decode(gen[0][enc["input_ids"].shape[1]:], skip_special_tokens=True).strip().lower()
    for c in CATEGORIES:
        if c in out:
            return c
    return None

def triage_accuracy(model, tokenizer, df, max_n=30):
    if model is None or tokenizer is None:
        return None
    correct = 0
    for _, row in df.head(max_n).iterrows():
        if predict_category(model, tokenizer, row["text"]) == row["category"]:
            correct += 1
    return correct / max_n


In [ ]:
acc_fp16 = triage_accuracy(model, tokenizer, tickets) if LIVE_FP16 else None
acc_8bit = triage_accuracy(model_8bit, tokenizer, tickets) if LIVE_QUANT else None
acc_4bit = triage_accuracy(model_4bit, tokenizer, tickets) if LIVE_QUANT else None

print("triage accuracy FP16:   ", acc_fp16)
print("triage accuracy 8-bit:  ", acc_8bit)
print("triage accuracy 4-bit:  ", acc_4bit)


## 9. The quality-vs-VRAM table

This is the artifact. Read the tradeoff directly: how much VRAM did we save, and what did
it cost in perplexity and triage accuracy?


In [ ]:
rows = [
    {"precision": "FP16",     "vram_gb": round(vram_bytes(PARAMS, 16)/1e9, 2),
     "perplexity": None if ppl_fp16 is None else round(ppl_fp16, 3), "triage_acc": acc_fp16},
    {"precision": "INT8",     "vram_gb": round(vram_bytes(PARAMS, 8)/1e9, 2),
     "perplexity": None if ppl_8bit is None else round(ppl_8bit, 3), "triage_acc": acc_8bit},
    {"precision": "INT4/NF4", "vram_gb": round(vram_bytes(PARAMS, 4)/1e9, 2),
     "perplexity": None if ppl_4bit is None else round(ppl_4bit, 3), "triage_acc": acc_4bit},
]
results = pd.DataFrame(rows)
print(results.to_string(index=False))


## 10. CPU fallback path (always runs)

If you are on CPU only, the measured cells above are `None`. This cell reproduces the
**documented** quality ladder from `reference/knowledge-base/06-model-engineering.md`, the general
trend (FP16 ≈ FP32 for inference, 8-bit ≈ negligible loss, 4-bit = small) expressed as
perplexity *ratios* and triage-accuracy *expectations*. These are **estimates, not
measurements**, the whole point of the live cells is to replace this row with real
numbers from *your* hardware.


In [ ]:
fallback = pd.DataFrame([
    {"precision": "FP16",     "vram_gb": round(vram_bytes(PARAMS, 16)/1e9, 2),
     "ppl_ratio_vs_fp16": 1.00, "triage_acc_estimate": 0.92},
    {"precision": "INT8",     "vram_gb": round(vram_bytes(PARAMS, 8)/1e9, 2),
     "ppl_ratio_vs_fp16": 1.01, "triage_acc_estimate": 0.92},
    {"precision": "INT4/NF4", "vram_gb": round(vram_bytes(PARAMS, 4)/1e9, 2),
     "ppl_ratio_vs_fp16": 1.06, "triage_acc_estimate": 0.89},
])
print("Documented quality ladder (perplexity ratio + expected triage accuracy):")
print(fallback.to_string(index=False))


## 11. Takeaway

The decision rule is the same in every path: **eval on the task, not on perplexity
alone.** A quant is acceptable when it keeps your task metric within your tolerance at a
fraction of the memory and cost. Perplexity screens candidates; the triage accuracy,
and, in Week 9's second notebook, the serving benchmark, decides.


In [ ]:
# FINAL number: quality retention of the smallest quant that actually ran
# (or the documented estimate) relative to FP16. 100 = no loss.
if acc_fp16 is not None and acc_4bit is not None:
    retention = 100.0 * (acc_4bit / acc_fp16)
    source = "measured (live)"
elif acc_fp16 is not None:
    retention = 100.0
    source = "FP16-only (no live quant on this device)"
else:
    fp16_est = fallback.loc[fallback.precision == "FP16", "triage_acc_estimate"].iloc[0]
    q4_est   = fallback.loc[fallback.precision == "INT4/NF4", "triage_acc_estimate"].iloc[0]
    retention = 100.0 * (q4_est / fp16_est)
    source = "estimated (CPU fallback)"

print(f"QUALITY_RETENTION_PCT={retention:.2f}   ({source})")
